# 03b — PyCaret time_series con distintos horizontes temporales

SAB.MC diario + IBEX mensual + EUR/USD trimestral combinados con `pd.merge_asof(direction='backward')`. Réplica del patrón del notebook de prácticas `forecast_distinto_horizonte_temporal.ipynb`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow

from src.data_loader import get_sab, get_ibex, get_eurusd

In [ ]:
sab = get_sab(period='5y')
ibex = get_ibex(period='5y')
eurusd = get_eurusd(period='5y')
for d in (sab, ibex, eurusd):
    d['Date'] = pd.to_datetime(d['Date'])
print('SAB:', len(sab), 'IBEX:', len(ibex), 'EURUSD:', len(eurusd))

In [ ]:
# diario
diario = pd.DataFrame({'Fecha': sab['Date'], 'close_sab': sab['Close']})
# mensual (fin de mes)
mes = (ibex.set_index('Date')['Close'].resample('M').last().rename('close_ibex').reset_index().rename(columns={'Date':'Fecha'}))
# trimestral (fin de trimestre)
trim = (eurusd.set_index('Date')['Close'].resample('Q').last().rename('close_eurusd').reset_index().rename(columns={'Date':'Fecha'}))
print('Diario:', len(diario), '| Mensual:', len(mes), '| Trimestral:', len(trim))

In [ ]:
# merge_asof backward (sin look-ahead leak)
dm = pd.merge_asof(diario.sort_values('Fecha'), mes.sort_values('Fecha'), on='Fecha', direction='backward')
df = pd.merge_asof(dm.sort_values('Fecha'), trim.sort_values('Fecha'), on='Fecha', direction='backward')
df = df.dropna().reset_index(drop=True)
df.tail()

In [ ]:
df.set_index('Fecha')['close_sab'].plot(figsize=(11,4), title='SAB.MC — cierre diario')
plt.ylabel('EUR'); plt.tight_layout(); plt.show()

## PyCaret `time_series`

`fh=90` (~4.5 meses bursátiles), `seasonal_period=5` (semana bursátil — ajustamos respecto al notebook de prácticas, que usaba 'D' sobre datos continuos sintéticos).

In [ ]:
from pycaret.time_series import TSForecastingExperiment

st = TSForecastingExperiment()
st.setup(
    data=df,
    target='close_sab',
    session_id=100,
    fh=90,
    fold=3,
    seasonal_period=5,
    ignore_features=['Fecha'],
    numeric_imputation_target='ffill',
    verbose=False,
)

In [ ]:
best = st.compare_models(sort='MASE', n_select=1, exclude=['prophet','bats','tbats','lar_cds_dt'], turbo=True, verbose=False)
leaderboard = st.pull()
leaderboard.head(10)

In [ ]:
print('Modelo ganador:', type(best).__name__)
holdout_pred = st.predict_model(best)
holdout_pred.head()

In [ ]:
y_pred = holdout_pred.iloc[:, 0]
# valor real correspondiente a esas fechas (alineamos por posición — últimos 90 puntos del histórico)
ts_real = df.set_index('Fecha')['close_sab'].iloc[-90:]
y_pred_vals = y_pred.values[:len(ts_real)]
y_true_vals = ts_real.values[:len(y_pred_vals)]
mae  = float(np.abs(y_true_vals - y_pred_vals).mean())
rmse = float(np.sqrt(((y_true_vals - y_pred_vals)**2).mean()))
mape = float((np.abs(y_true_vals - y_pred_vals) / y_true_vals).mean() * 100)
print(f'MULTI-HORIZONTE  {type(best).__name__}  MAE: {mae:.3f}  RMSE: {rmse:.3f}  MAPE: {mape:.2f}%')

In [ ]:
# MLflow
mlflow.set_tracking_uri('file:///' + (ROOT / 'mlruns').as_posix())
mlflow.set_experiment('sab_forecast_pycaret_horizontes')

with mlflow.start_run(run_name=f'multihorizon_{type(best).__name__}'):
    mlflow.log_param('target', 'close_sab')
    mlflow.log_param('fh', 90)
    mlflow.log_param('regresores', 'close_ibex (mensual), close_eurusd (trimestral)')
    mlflow.log_param('estimator', type(best).__name__)
    mlflow.log_metric('mae', mae)
    mlflow.log_metric('rmse', rmse)
    mlflow.log_metric('mape', mape)

In [ ]:
# Guardar el forecast del holdout multi-horizonte para la comparativa.
# NOTA: no proyectamos al futuro porque el modelo usa regresores exogenos
# (IBEX, EUR/USD) y predict_model(fh=90) exigiria valores FUTUROS de esos
# regresores, que son desconocidos. El valor del modelo multi-horizonte
# esta en la metrica de holdout (comparable), no en una proyeccion ciega.
fechas = ts_real.index[:len(y_pred_vals)]
fcst_out = pd.DataFrame({'ds': fechas, 'yhat': y_pred_vals})
fcst_path = ROOT / 'data' / 'sab_forecast_pycaret_horizontes_90d.csv'
fcst_out.to_csv(fcst_path, index=False)
print(f'Forecast holdout multi-horizonte guardado: {fcst_path} ({len(fcst_out)} filas)')